# Telco Customer Churn Prediction

**Goal:** predict which telecom customers are likely to leave (churn), and find out what drives it.

**Pipeline:** load data → clean → split → train models → check results → business findings

Data: Kaggle "Telco Customer Churn" (`WA_Fn-UseC_-Telco-Customer-Churn.csv`). Run the cells from top to bottom.

## Step 1: Load the data
Run the next cell and choose the CSV file when the upload window opens. (Colab deletes uploaded files when a session resets, so do this first each time.)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 2: Explore the data
How many rows and columns, what types, and how many customers churn?

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


About 27% of customers churn (1,869 of 7,043). The data is imbalanced, so accuracy alone is misleading: a model that always says "stays" would be about 73% accurate and useless.

## Step 3: Clean the data
`TotalCharges` is stored as text because 11 new customers (tenure = 0) have a blank value. They have not been billed yet, so we fill the blanks with 0 and convert the column to numbers.

In [ ]:
df[df['TotalCharges'] == ' ']

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df['TotalCharges'].dtype

dtype('float64')

Convert the target column to numbers: Yes = 1 (churned), No = 0 (stayed). Then turn the other text columns into True/False columns (one-hot encoding), because models need numbers.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df_encoded = pd.get_dummies(df, columns=['gender', 'Partner', 'Dependents', 'PhoneService',
                                         'MultipleLines', 'InternetService', 'OnlineSecurity',
                                         'OnlineBackup', 'DeviceProtection', 'TechSupport',
                                         'StreamingTV', 'StreamingMovies', 'Contract',
                                         'PaperlessBilling', 'PaymentMethod'])

# customerID is just a label, it does not help predict churn
df_encoded = df_encoded.drop('customerID', axis=1)
df_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,True,False,False,True,True,...,False,True,False,False,False,True,False,False,True,False
1,0,34,56.95,1889.50,0,False,True,True,False,True,...,False,False,True,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,False,True,True,False,True,...,False,True,False,False,False,True,False,False,False,True
3,0,45,42.30,1840.75,0,False,True,True,False,True,...,False,False,True,False,True,False,True,False,False,False
4,0,2,70.70,151.65,1,True,False,True,False,True,...,False,True,False,False,False,True,False,False,True,False


## Step 4: Split into clues (X), answer (y), and train/test sets
80% of customers teach the model, 20% are held back as an exam it has not seen.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('Churn', axis=1)   # clues
y = df_encoded['Churn']                # the answer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(5634, 45)
(1409, 45)


## Step 5: Model 1, Logistic Regression (baseline)
Features are scaled first, because columns like `TotalCharges` (thousands) and the True/False columns are on very different scales.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, predictions))
print()
print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Accuracy: 0.8204400283889283

[[933 103]
 [150 223]]

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.68      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.82      1409



Reading the results for churners (row 1): **recall** is how many of the real churners the model finds, and **precision** is how many of the customers it flags really churn. The baseline finds about 60% of churners.

## Step 6: Model 2, Random Forest
Many decision trees that vote. Trees do not need scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_predictions))
print()
print(confusion_matrix(y_test, rf_predictions))
print()
print(classification_report(y_test, rf_predictions))

Accuracy: 0.794889992902768

[[947  89]
 [200 173]]

              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409



The Random Forest did **worse** than the simple baseline: it finds only about 46% of churners. A more complex model is not automatically better.

## Step 7: Model 3, balanced Logistic Regression
`class_weight='balanced'` makes a mistake on a churner count for more, so the model pays more attention to them.

In [ ]:
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_balanced.fit(X_train_scaled, y_train)
lr_predictions = lr_balanced.predict(X_test_scaled)

print(confusion_matrix(y_test, lr_predictions))
print()
print(classification_report(y_test, lr_predictions))

[[746 290]
 [ 66 307]]

              precision    recall  f1-score   support

           0       0.92      0.72      0.81      1036
           1       0.51      0.82      0.63       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



Recall for churners rises to about 0.82 (it finds 82 of every 100 real churners). The price is lower precision (about 0.51), so more false alarms. Which model is best is a business decision: it depends on what a missed churner costs compared with an unnecessary retention offer.

## Step 7b: Model 4, XGBoost
Trees are built one after another, and each new tree tries to fix the mistakes of the previous ones. `scale_pos_weight` gives churners more weight, like `class_weight='balanced'` did for Logistic Regression.

If you see `No module named 'xgboost'`, run `!pip install xgboost` in a cell first.

In [ ]:
from xgboost import XGBClassifier

weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(scale_pos_weight=weight, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

print(confusion_matrix(y_test, xgb_predictions))
print()
print(classification_report(y_test, xgb_predictions))

[[821 215]
 [105 268]]

              precision    recall  f1-score   support

           0       0.89      0.79      0.84      1036
           1       0.55      0.72      0.63       373

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409



XGBoost lands in the middle: recall about 0.72 (it finds 72 of every 100 leavers) with precision about 0.55. The F1 scores of the weighted models are almost tied (about 0.63), so the choice depends on what a missed leaver costs compared with an unnecessary discount.

## Step 8: What drives churn?
Positive numbers push toward churn, negative numbers push toward staying (features were scaled, so sizes can be compared).

In [ ]:
importance = pd.Series(lr_balanced.coef_[0], index=X.columns)
print(importance.sort_values(ascending=False).head(8))
print()
print(importance.sort_values().head(8))

TotalCharges                      0.604858
InternetService_Fiber optic       0.382311
Contract_Month-to-month           0.327878
StreamingMovies_Yes               0.183298
StreamingTV_Yes                   0.157071
PaymentMethod_Electronic check    0.124069
OnlineSecurity_No                 0.115300
TechSupport_No                    0.095035
dtype: float64

tenure                                 -1.241805
MonthlyCharges                         -0.858748
Contract_Two year                      -0.371835
InternetService_DSL                    -0.305267
DeviceProtection_No internet service   -0.108866
OnlineBackup_No internet service       -0.108866
OnlineSecurity_No internet service     -0.108866
TechSupport_No internet service        -0.108866
dtype: float64


Check the main patterns directly in the data. `tenure`, `MonthlyCharges` and `TotalCharges` overlap, so do not over-read their individual signs.

In [ ]:
print(df.groupby('Contract')['Churn'].mean())
print()
print(df.groupby('InternetService')['Churn'].mean())

Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn, dtype: float64

InternetService
DSL            0.189591
Fiber optic    0.418928
No             0.074050
Name: Churn, dtype: float64


## Step 9: Business findings and impact

- Customers on **month-to-month** contracts churn at about 42.7%, against 2.8% for two-year contracts.
- **Fiber optic** customers churn at about 41.9%, against 19.0% for DSL and 7.4% for no internet.
- Low **tenure** (new customers) is the strongest churn signal.

**Example impact (made-up numbers to show scale):** if a plan saves 10% of the 1,869 churners, that is about 187 customers. At an example value of 65 per month (780 per year), that protects about 145,860 in yearly income.

In [ ]:
churners = 1869
saved = churners * 0.10
yearly_value = 65 * 12
print("Customers kept:", round(saved))
print("Yearly income protected:", round(saved) * yearly_value)

Customers kept: 187
Yearly income protected: 145860


## Step 10: SQL analysis

The same questions answered with SQL, using Python's built-in `sqlite3`. `df` (the cleaned data, with Churn as 1/0) is saved as a table named `churn`. Because Churn is 0 or 1, `AVG(Churn)` is the churn rate.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")

def sql(query):
    return pd.read_sql_query(query, conn)

df.to_sql("churn", conn, index=False, if_exists="replace")

sql("""
SELECT Contract, COUNT(*) AS customers, ROUND(AVG(Churn), 3) AS churn_rate
FROM churn
GROUP BY Contract
""")

,Contract,customers,churn_rate
0,Month-to-month,3875,0.427
1,One year,1473,0.113
2,Two year,1695,0.028


Churn rate by contract and internet type. `HAVING COUNT(*) > 500` keeps only groups large enough to trust (`WHERE` filters rows before grouping, `HAVING` filters groups after).

In [ ]:
sql("""
SELECT Contract, InternetService, COUNT(*) AS customers, ROUND(AVG(Churn), 3) AS churn_rate
FROM churn
GROUP BY Contract, InternetService
HAVING COUNT(*) > 500
ORDER BY churn_rate DESC
""")

,Contract,InternetService,customers,churn_rate
0,Month-to-month,Fiber optic,2128,0.546
1,Month-to-month,DSL,1223,0.322
2,One year,Fiber optic,539,0.193
3,Month-to-month,No,524,0.189
4,One year,DSL,570,0.093
5,Two year,DSL,628,0.019
6,Two year,No,638,0.008


Month-to-month customers with fiber optic churn the most (about 54.6%, 2,128 customers). Is it the price? Compare average charges with churn for each group.

In [ ]:
sql("""
SELECT Contract, InternetService,
       ROUND(AVG(MonthlyCharges), 2) AS avg_charges,
       ROUND(AVG(Churn), 3) AS churn_rate
FROM churn
GROUP BY Contract, InternetService
ORDER BY churn_rate DESC
""")

,Contract,InternetService,avg_charges,churn_rate
0,Month-to-month,Fiber optic,87.02,0.546
1,Month-to-month,DSL,50.22,0.322
2,One year,Fiber optic,98.78,0.193
3,Month-to-month,No,20.41,0.189
4,One year,DSL,61.40,0.093
5,Two year,Fiber optic,104.57,0.072
6,One year,No,20.82,0.025
7,Two year,DSL,70.46,0.019
8,Two year,No,21.78,0.008


Price alone does not explain it: fiber customers on two-year contracts pay the most (about 104.57) and almost never leave (about 7.2%). Next, a clue about service: tech support, among fiber optic customers only.

In [ ]:
sql("""
SELECT TechSupport, COUNT(*) AS customers, ROUND(AVG(Churn), 3) AS churn_rate
FROM churn
WHERE InternetService = 'Fiber optic'
GROUP BY TechSupport
""")

,TechSupport,customers,churn_rate
0,No,2230,0.494
1,Yes,866,0.226


Fiber customers without tech support churn at about 49.4%, against 22.6% with it. But that gap mixes the contract types, so check inside each contract before trusting it.

In [ ]:
sql("""
SELECT Contract, TechSupport, COUNT(*) AS customers, ROUND(AVG(Churn), 3) AS churn_rate
FROM churn
WHERE InternetService = 'Fiber optic'
GROUP BY Contract, TechSupport
ORDER BY Contract, TechSupport
""")

,Contract,TechSupport,customers,churn_rate
0,Month-to-month,No,1796,0.575
1,Month-to-month,Yes,332,0.389
2,One year,No,313,0.185
3,One year,Yes,226,0.204
4,Two year,No,121,0.083
5,Two year,Yes,308,0.068


Step 11:
SQL cross-check with Excel and Power BI.

I rebuilt the main results in Excel and Power BI. Here is the churn rate by tenure band in SQL, using CASE WHEN to make the bands. Because Churn is 0 or 1, AVG(Churn) is the churn rate.

In [ ]:
sql("""
SELECT
  CASE
    WHEN tenure BETWEEN 0 AND 11 THEN '0-11'
    WHEN tenure BETWEEN 12 AND 23 THEN '12-23'
    WHEN tenure BETWEEN 24 AND 35 THEN '24-35'
    WHEN tenure BETWEEN 36 AND 47 THEN '36-47'
    WHEN tenure BETWEEN 48 AND 59 THEN '48-59'
    ELSE '60-72'
  END AS tenure_band,
  COUNT(*) AS customers,
  ROUND(AVG(Churn), 3) AS churn_rate
FROM churn
GROUP BY tenure_band
ORDER BY MIN(tenure)
""")

,tenure_band,customers,churn_rate
0,0-11,2069,0.483
1,12-23,1047,0.295
2,24-35,876,0.220
3,36-47,748,0.195
4,48-59,820,0.150
5,60-72,1483,0.067


Which internet service churns most inside each contract? A window function ranks the rows inside each contract (PARTITION BY) and keeps every row.

In [ ]:
sql("""
SELECT Contract, InternetService, customers, churn_rate,
       RANK() OVER (PARTITION BY Contract ORDER BY churn_rate DESC) AS rank_in_contract
FROM (
    SELECT Contract, InternetService, COUNT(*) AS customers, ROUND(AVG(Churn), 3) AS churn_rate
    FROM churn
    GROUP BY Contract, InternetService
)
ORDER BY Contract, rank_in_contract
""")

,Contract,InternetService,customers,churn_rate,rank_in_contract
0,Month-to-month,Fiber optic,2128,0.546,1
1,Month-to-month,DSL,1223,0.322,2
2,Month-to-month,No,524,0.189,3
3,One year,Fiber optic,539,0.193,1
4,One year,DSL,570,0.093,2
5,One year,No,364,0.025,3
6,Two year,Fiber optic,429,0.072,1
7,Two year,DSL,628,0.019,2
8,Two year,No,638,0.008,3


### SQL findings

- **Month-to-month fiber customers:** those without tech support churn at about 57.5% (1,796 customers), against about 38.9% with it (332 customers).
- **One-year contracts:** no benefit (about 18.5% without, 20.4% with). **Two-year contracts:** a small gap in small groups (about 8.3% without, 6.8% with), so I would not read much into it.
- Part of the first, larger gap (49.4% against 22.6%) came from the contract mix: most fiber customers without tech support are on month-to-month contracts, and those customers churn more anyway. Checking inside each contract separates the two.
- The data cannot measure how good the tech support is, and it cannot show cause and effect. I would recommend testing a free tech support trial for month-to-month fiber customers and checking whether churn falls.